# DS 4300: Building a Data Lakehouse on AWS, Part 2

The OneShop hands-on from the lecture: files in **S3**, a catalog in **Glue**, SQL in **Athena**, and real tables with **Iceberg**.

Every step from the lecture is filled in. Run the cells top to bottom.

A `!` at the start of a line runs it as a shell command, so each `!aws ...` line is the same command you would type in `docker exec -it spark-iceberg bash`. The endpoint and credentials are already set in this container.

Run this notebook once on a fresh stack. Running it again makes `mb`, `create-database`, `CREATE EXTERNAL TABLE`, and `CREATE TABLE` fail because those resources already exist, and the `INSERT` would add the nine rows a second time. To start over, run `docker compose down` and then `docker compose up -d`.

## Step 3: Proof of life (STS)

STS answers "who am I, and which account am I using?" Expect `"Account": "000000000000"`.

In [ ]:
!aws sts get-caller-identity

## Step 4: Amazon S3, the storage layer

Make a bucket, copy the sample users file into it, list the bucket, and print the object to prove it arrived intact.

This notebook runs in `notebooks/`, so the data file is at `../data/users.csv`. In the container shell the path is `data/users.csv`.

In [ ]:
!aws s3 mb s3://oneshop-landing

In [ ]:
!aws s3 cp ../data/users.csv s3://oneshop-landing/users/users.csv

In [ ]:
!aws s3 ls s3://oneshop-landing --recursive

In [ ]:
# "-" as the destination prints the object to the screen
!aws s3 cp s3://oneshop-landing/users/users.csv -

## Step 5: AWS Glue, the catalog layer

Glue stores table names, column types, and S3 locations, never the data itself.

In [ ]:
# ["accounts"], made by the startup script
!aws glue get-databases --query 'DatabaseList[].Name'

In [ ]:
!aws glue create-database --database-input '{"Name":"oneshop","LocationUri":"s3://warehouse/oneshop"}'

In [ ]:
# [] (nothing registered yet)
!aws glue get-tables --database-name oneshop --query 'TableList[].Name'

## Step 6: Amazon Athena, the query engine layer

Every query takes three steps: **start** it (`start-query-execution`), **check** it (`get-query-execution`), then **fetch** the rows (`get-query-results`).

`qid = !aws ...` saves the command's output in the Python variable `qid`, so later cells can pass the QueryExecutionId back with `{qid}`.

### Describe the CSV as a table

An external table is only a definition stored in Glue. The data stays in the CSV in `oneshop-landing`.

In [ ]:
sql = """CREATE EXTERNAL TABLE users_raw
(id int, first_name string, last_name string, email string)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
LOCATION 's3://oneshop-landing/users/'
TBLPROPERTIES ('skip.header.line.count'='1')"""

qid = !aws athena start-query-execution --query-execution-context Database=oneshop --query-string "{sql}" --query QueryExecutionId --output text
qid = qid[0]
qid

In [ ]:
# Run again until it says "SUCCEEDED"
!aws athena get-query-execution --query-execution-id {id} --query QueryExecution.Status.State

In [ ]:
# Now shows users_raw. The CREATE wrote a Glue entry; the CSV is untouched.
!aws glue get-tables --database-name oneshop --query 'TableList[].Name'

### Run a query and fetch the rows

Fetch too early and you get `InvalidRequestException: Query has not succeeded yet`. The query is still running in the background, so check its status and fetch again.

In [ ]:
sql = "SELECT id, last_name, email FROM users_raw ORDER BY last_name"

qid = !aws athena start-query-execution --query-execution-context Database=oneshop --query-string "{sql}" --query QueryExecutionId --output text
qid = qid[0]
qid

In [ ]:
!aws athena get-query-execution --query-execution-id {qid} --query QueryExecution.Status.State

In [ ]:
!aws athena get-query-results --query-execution-id {qid} --query 'ResultSet.Rows[].Data[*].VarCharValue' --output table

### The same three steps as one Python function

`athena(sql)` makes the same three calls through boto3, the AWS SDK for Python: start, check until done, fetch. It returns the rows as a pandas DataFrame, and every value comes back as text. Step 7 uses it so each statement runs in one cell; the three `!aws athena` commands above do the same thing.

In [ ]:
import time

import boto3
import pandas as pd

athena_client = boto3.client("athena")


def athena(sql, database="oneshop"):
    """Start a query, wait for it to finish, and return its rows as a DataFrame."""
    qid = athena_client.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
    )["QueryExecutionId"]
    while True:
        status = athena_client.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if status["State"] not in ("QUEUED", "RUNNING"):
            break
        time.sleep(0.5)
    if status["State"] != "SUCCEEDED":
        raise RuntimeError(f"{status['State']}: {status.get('StateChangeReason')}")
    rows = athena_client.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    values = [[col.get("VarCharValue") for col in row["Data"]] for row in rows]
    return pd.DataFrame(values[1:], columns=values[0]) if values else pd.DataFrame()


athena("SELECT id, last_name, email FROM users_raw ORDER BY last_name")

## Step 7: Apache Iceberg, the table format layer

An Iceberg table is **files in S3** plus a **pointer in Glue** to the latest version. Every commit moves the pointer.

Each SQL statement from slides 38 and 40 runs through `athena(sql)` in the `oneshop` database: the same start, check, fetch pattern as step 6, waiting for `SUCCEEDED` before the next statement.

These work on real AWS but fail in local Floci:
- `CREATE TABLE ... AS SELECT` reports SUCCEEDED but creates nothing. Use `CREATE TABLE` and then `INSERT ... SELECT`.
- `SHOW TABLES` and `DESCRIBE` fail. Use `aws glue get-tables` and `aws glue get-table`.
- `ALTER TABLE ... ADD COLUMNS`, the `"table$snapshots"` metadata tables, `FOR TIMESTAMP AS OF`, and `OPTIMIZE` fail. Goals 7.6 and 7.7 use PyIceberg for snapshots and time travel.

### 7.1 Create the Iceberg table

**Goal:** create an empty Iceberg table `users` with the same four columns as `users_raw` (`id int, first_name string, last_name string, email string`), stored at `s3://warehouse/oneshop/users/`. The `TBLPROPERTIES ('table_type'='ICEBERG')` clause is what makes it Iceberg rather than a plain external table.

In [ ]:
athena("""CREATE TABLE users (id int, first_name string, last_name string, email string)
LOCATION 's3://warehouse/oneshop/users/'
TBLPROPERTIES ('table_type'='ICEBERG')""")

### 7.2 Load it

**Goal:** copy the nine rows from `users_raw` into `users` with `INSERT INTO ... SELECT`, then query `users` to confirm they arrived.

In [ ]:
athena("INSERT INTO users SELECT * FROM users_raw")
athena("SELECT * FROM users ORDER BY id")

### 7.3 Look at what Iceberg wrote to S3

**Goal:** list everything under `s3://warehouse/oneshop/users/`. Match the files to the chain from the lecture: `data/*.parquet` (the rows), `metadata/*.metadata.json` (schema and snapshot list), `snap-*.avro` (manifest list), and `*-m0.avro` (manifest).

In [ ]:
# data/*.parquet + metadata/* files
!aws s3 ls s3://warehouse/oneshop/users/ --recursive

### 7.4 Find the pointer in Glue

**Goal:** show the Glue table parameters for `oneshop.users`. Confirm `table_type` is `ICEBERG` and note the number at the start of the `metadata_location` file name (`00001`).

In [ ]:
# "table_type": "ICEBERG", "metadata_location": ".../metadata/00001-....metadata.json"
!aws glue get-table --database-name oneshop --name users --query Table.Parameters

### 7.5 Change one row, then check the pointer

**Goal:** set the email of the user with `id = 1` to `juli@example.com` with an `UPDATE`. Then ask Glue for just `metadata_location` again: it should now point to a `00002-...metadata.json` file. List the S3 files again and notice that the old metadata file is still there.

In [ ]:
athena("UPDATE users SET email = 'juli@example.com' WHERE id = 1")

# now .../00002-....metadata.json
!aws glue get-table --database-name oneshop --name users --query Table.Parameters.metadata_location --output text

# 00001 is still there next to 00002: the old version is kept
!aws s3 ls s3://warehouse/oneshop/users/metadata/

### 7.6 List the table's snapshots (PyIceberg)

**Goal:** load `oneshop.users` from the `glue` catalog with PyIceberg (`from pyiceberg.catalog import load_catalog`) and list its snapshots with their operation and commit time. Expect two: an `append` from the INSERT and an `overwrite` from the UPDATE.

In [ ]:
from pyiceberg.catalog import load_catalog

users = load_catalog("glue").load_table("oneshop.users")
snapshots = users.inspect.snapshots().to_pandas().sort_values("committed_at")
snapshots[["snapshot_id", "operation", "committed_at"]]

### 7.7 Time travel (PyIceberg)

**Goal:** read the row with `id = 1` as it was in the **first** snapshot, then as it is now. The old snapshot still shows `jarthars0@mit.edu`; the current table shows `juli@example.com`.

In [ ]:
first_snapshot = int(snapshots["snapshot_id"].iloc[0])

print("As of the first snapshot (after the INSERT):")
display(users.scan(snapshot_id=first_snapshot, row_filter="id = 1").to_pandas())

print("Now (after the UPDATE):")
display(users.scan(row_filter="id = 1").to_pandas())

## Shutting down

- `docker compose stop` and later `docker compose start` keep the data.
- `docker compose down` stops everything and resets the emulated data: the `oneshop-landing` bucket, the `oneshop` database, and its tables are gone. The startup script recreates only `warehouse`, `athena-results`, and `accounts`.